In [ ]:
import os
import csv
import shutil

DATA_PATH = './data_BSAFusion'
CSV_PATH = './data_BSAFusion/CSV_FILES' 
OUTPUT_PATH = "My_Dataset" 
LOG_FILE = os.path.join(OUTPUT_PATH, "processing_errors.log") 

def create_dataset():
    with open(LOG_FILE, "w") as log_file:
        for csv_name in ["CT_MRI.csv", "PET_MRI.csv", "SPECT_MRI.csv"]:
            combo_name = csv_name.replace("_", "-")  
            mod1, mod2 = combo_name.split("-") 
            
            csv_path = os.path.join(CSV_PATH, csv_name)
            with open(csv_path, "r") as f:
                reader = csv.reader(f)
                headers = next(reader)
                
                for row_num, row in enumerate(reader, start=2):  
                    if row[0].strip():
                        _process_file(
                            combo_name, [mod1, mod2], 
                            row[0].strip(), "train", 
                            log_file, row_num
                        )
                    if len(row) > 1 and row[1].strip():
                        _process_file(
                            combo_name, [mod1, mod2],
                            row[1].strip(), "test",
                            log_file, row_num
                        )

def _process_file(combo, mods, filename, split_type, log, csv_row):
    for mod in mods:
        src = os.path.join(DATA_PATH, combo, mod, filename)
        dest_dir = os.path.join(OUTPUT_PATH, combo, split_type, mod)
        os.makedirs(dest_dir, exist_ok=True)
        
        try:
            if os.path.exists(src):
                shutil.copy2(src, dest_dir)
            else:
                raise FileNotFoundError(f"File not found: {src}")
        except Exception as e:
            log.write(
                f"[ERROR] {combo} | CSV Row {csv_row} | "
                f"{split_type}/{mod}/{filename} | {str(e)}\n"
            )

if __name__ == "__main__":
    print("🚀 Creating dataset")
    
    if os.path.exists(OUTPUT_PATH):
        shutil.rmtree(OUTPUT_PATH)
    os.makedirs(OUTPUT_PATH, exist_ok=True)
    
    create_dataset()
    
    print(f"✅ Finished, saved at {os.path.abspath(OUTPUT_PATH)}")
    print(f"⚠️ Check logs: {os.path.abspath(LOG_FILE)}")
